In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
from collections import defaultdict
import json
import os
import random
from tqdm import tqdm

# 1. Check the computing device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current computation device: {device}")

# 2. Load the dataset
print("Loading dataset...")
train_df = pd.read_json('/content/train_indexed.jsonl', lines=True)
test_df = pd.read_json('/content/test_indexed.jsonl', lines=True)

num_users = max(train_df['user_idx'].max(), test_df['user_idx'].max()) + 1
num_items = max(train_df['item_idx'].max(), test_df['item_idx'].max()) + 1

# 3. Define the data loader and network structure
class AmazonDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.items = torch.tensor(df['item_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
    def __len__(self): return len(self.ratings)
    def __getitem__(self, idx): return self.users[idx], self.items[idx], self.ratings[idx]

train_loader = DataLoader(AmazonDataset(train_df), batch_size=1024, shuffle=True)
test_loader = DataLoader(AmazonDataset(test_df), batch_size=1024, shuffle=False)

class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(MatrixFactorization, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        u_emb = self.user_embedding(user)
        i_emb = self.item_embedding(item)
        dot = (u_emb * i_emb).sum(1)
        return dot + self.user_bias(user).squeeze() + self.item_bias(item).squeeze() + self.global_bias

model = MatrixFactorization(num_users, num_items, embedding_dim=64).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)

# 4. Training Phase (Max 30 Epochs)

epochs = 30
print(f"\n Starting full training for Baseline Model 1 (Collaborative Filtering), total {epochs} epochs ...")

# ----------------- EARLY STOPPING CONFIGURATION -----------------
# Set the tolerance to 3 evaluation cycles (i.e. 15 rounds).
# Given the shallow-layer nature of matrix factorisation models, a longer convergence period is required.
patience = 3
best_rmse = float('inf')
patience_counter = 0
# ----------------------------------------------------------------

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        model.eval()
        test_preds, test_trues = [], []
        with torch.no_grad():
            for users, items, ratings in test_loader:
                users, items = users.to(device), items.to(device)
                test_preds.extend(model(users, items).cpu().numpy())
                test_trues.extend(ratings.numpy())
        rmse = math.sqrt(mean_squared_error(test_trues, test_preds))
        print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {total_loss/len(train_loader):.4f} | Test RMSE: {rmse:.4f}")

        # Early termination logic check
        if rmse < best_rmse:
            best_rmse = rmse
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f" Early stopping triggered at epoch {epoch+1}.")
                break

mae = mean_absolute_error(test_trues, test_preds)

# Save model weights
model_save_path = '/content/baseline1_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"\n Model weights successfully saved to: {model_save_path}")

# 5. Two-track evaluation: Full-ranking and Sampled-ranking

# ----------------- Track A: Full Top-10 Evaluation -----------------
print("\n[Track A] Performing FULL Test Set Top-10 Recommendation Evaluation...")
user_true_items = defaultdict(set)
for _, row in test_df.iterrows():
    user_true_items[int(row['user_idx'])].add(int(row['item_idx']))

all_items_list = list(set(train_df['item_idx'].unique()).union(set(test_df['item_idx'].unique())))
all_items_tensor = torch.tensor(all_items_list, dtype=torch.long).to(device)

precisions_full, recalls_full, ndcgs_full = [], [], []
test_users_full = list(user_true_items.keys())

model.eval()
with torch.no_grad():
    for i, user in enumerate(test_users_full):
        if (i+1) % 5000 == 0:
            print(f"  Progress: Evaluated {i+1} / {len(test_users_full)} users...")

        true_items = user_true_items[user]
        user_tensor = torch.tensor([user] * len(all_items_list), dtype=torch.long).to(device)
        preds = model(user_tensor, all_items_tensor).cpu().numpy()

        top_k_indices = np.argsort(preds)[::-1][:10]
        top_k_items = np.array(all_items_list)[top_k_indices]

        hits = [1 if item in true_items else 0 for item in top_k_items]
        hits_count = sum(hits)

        precisions_full.append(hits_count / 10.0)
        recalls_full.append(hits_count / len(true_items))

        dcg = sum([hit / math.log2(idx + 2) for idx, hit in enumerate(hits)])
        idcg = sum([1 / math.log2(idx + 2) for idx in range(min(len(true_items), 10))])
        ndcgs_full.append(dcg / idcg if idcg > 0 else 0)

final_precision_full = np.mean(precisions_full)
final_recall_full = np.mean(recalls_full)
final_ndcg_full = np.mean(ndcgs_full)

print(f" Full-ranking Precision@10: {final_precision_full:.6f}")
print(f" Full-ranking Recall@10: {final_recall_full:.6f}")
print(f" Full-ranking NDCG@10: {final_ndcg_full:.6f}")


# ----------------- Track B: Fair Sampled Top-10 Evaluation (Target + 99 Negatives) -----------------
print("\n[Track B] Performing FAIR Sampled Top-10 Evaluation (Target + 99 Negatives)...")

train_user_items = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
test_user_items = test_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

def calculate_ndcg_sampled(recommended_list, interacted_set):
    dcg = sum([1.0 / math.log2(i + 2) for i, item in enumerate(recommended_list) if item in interacted_set])
    idcg = sum([1.0 / math.log2(i + 2) for i in range(min(len(interacted_set), len(recommended_list)))])
    return dcg / idcg if idcg > 0 else 0.0

random.seed(42)
evaluate_users_sampled = random.sample(test_users_full, min(1000, len(test_users_full)))

total_precision_samp = 0.0
total_recall_samp = 0.0
total_ndcg_samp = 0.0
valid_users_count = 0

with torch.no_grad():
    for u in tqdm(evaluate_users_sampled, desc="Evaluating Sampled Users"):
        true_items = test_user_items.get(u, set())
        if not true_items: continue

        seen_items = train_user_items.get(u, set()).union(true_items)
        negative_items = set()
        while len(negative_items) < 99:
            rand_item = random.choice(all_items_list)
            if rand_item not in seen_items:
                negative_items.add(rand_item)

        candidate_items = list(true_items) + list(negative_items)
        u_tensor = torch.tensor([u] * len(candidate_items), dtype=torch.long).to(device)
        i_tensor = torch.tensor(candidate_items, dtype=torch.long).to(device)

        scores = model(u_tensor, i_tensor).cpu().numpy()
        item_score_pairs = list(zip(candidate_items, scores))
        item_score_pairs.sort(key=lambda x: x[1], reverse=True)
        top_10_items = [pair[0] for pair in item_score_pairs[:10]]

        hits = len(set(top_10_items) & true_items)
        total_precision_samp += hits / 10.0
        total_recall_samp += hits / len(true_items)
        total_ndcg_samp += calculate_ndcg_sampled(top_10_items, true_items)
        valid_users_count += 1

final_precision_samp = total_precision_samp / valid_users_count
final_recall_samp = total_recall_samp / valid_users_count
final_ndcg_samp = total_ndcg_samp / valid_users_count

print(f" Sampled Precision@10: {final_precision_samp:.4f}")
print(f" Sampled Recall@10: {final_recall_samp:.4f}")
print(f" Sampled NDCG@10: {final_ndcg_samp:.4f}")


# 6. Save the two sets of results

log_path = '/content/first_experiment_results.json'

results_full = {
    "model_name": "Baseline 1 (Matrix Factorization) - Full Ranking",
    "RMSE": float(rmse),
    "MAE": float(mae),
    "Precision_10": float(final_precision_full),
    "Recall_10": float(final_recall_full),
    "NDCG_10": float(final_ndcg_full),
    "epochs": epochs,
    "eval_users": len(test_users_full)
}

results_sampled = {
    "model_name": "Baseline 1 (Matrix Factorization) - Fair Sampled",
    "RMSE": float(rmse),
    "MAE": float(mae),
    "Precision_10": float(final_precision_samp),
    "Recall_10": float(final_recall_samp),
    "NDCG_10": float(final_ndcg_samp),
    "epochs": epochs,
    "eval_users": valid_users_count,
    "Notes": "Evaluated via Target+99 Negatives on 1000 sampled users"
}

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        all_results = json.load(f)
    # Clean up old Baseline 1 data to avoid duplication
    all_results = [res for res in all_results if "Baseline 1" not in res.get("model_name", "")]
else:
    all_results = []

all_results.extend([results_full, results_sampled])

with open(log_path, 'w') as f:
    json.dump(all_results, f, indent=4)

print("\n Both Full and Sampled evaluation metrics have been updated in first_experiment_results.json!")

Current computation device: cuda
Loading dataset...

 Starting full training for Baseline Model 1 (Collaborative Filtering), total 30 epochs ...
Epoch 01/30 | Train Loss: 15.4308 | Test RMSE: 3.2559
Epoch 05/30 | Train Loss: 0.2245 | Test RMSE: 1.4685
Epoch 10/30 | Train Loss: 0.0623 | Test RMSE: 1.4015
Epoch 15/30 | Train Loss: 0.0702 | Test RMSE: 1.3260
Epoch 20/30 | Train Loss: 0.0684 | Test RMSE: 1.2613
Epoch 25/30 | Train Loss: 0.0679 | Test RMSE: 1.2276
Epoch 30/30 | Train Loss: 0.0697 | Test RMSE: 1.2217

 Model weights successfully saved to: /content/baseline1_model.pth

[Track A] Performing FULL Test Set Top-10 Recommendation Evaluation...
  Progress: Evaluated 5000 / 27338 users...
  Progress: Evaluated 10000 / 27338 users...
  Progress: Evaluated 15000 / 27338 users...
  Progress: Evaluated 20000 / 27338 users...
  Progress: Evaluated 25000 / 27338 users...
 Full-ranking Precision@10: 0.001255
 Full-ranking Recall@10: 0.007252
 Full-ranking NDCG@10: 0.004930

[Track B] Perfo

Evaluating Sampled Users: 100%|██████████| 1000/1000 [00:00<00:00, 2004.29it/s]

 Sampled Precision@10: 0.0294
 Sampled Recall@10: 0.1586
 Sampled NDCG@10: 0.1002

 Both Full and Sampled evaluation metrics have been updated in first_experiment_results.json!
